[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/096.%20The%20Resilient%20Network%20Design%20%28Robust%20Optimization%29/P96-Tier-7_executed.ipynb)

---


# 96. The Resilient Network Design: Human–AI Partnership
## Tier 7: The Centaur Model

### Purpose and scope

This notebook illustrates a human-in-the-loop decision-support process for a facility-network design problem. An AI component ranks candidate facilities using a transparent cost-benefit score. Human experts then review its recommendation from different professional perspectives, and the partnership records agreement, trust, and learning metrics.

The facility-ranking rule is a heuristic, not a robust-optimization model: the notebook does not formulate uncertain scenarios, worst-case objectives, or robustness constraints. The chapter title identifies the broader topic; the executable example here focuses on the human–AI partnership framework. Its rules are illustrative and deterministic. They do not represent observed expert judgments or validated behavioral findings.

### Inputs, outputs, and assumptions

- **Inputs:** facility locations, capacities, fixed costs, strategic-importance labels, demand-zone locations and demand, and profiles for human experts and AI agents.
- **Outputs:** an AI facility recommendation, its service and cost indicators, expert feedback, a collaboration session, and partnership metrics.
- Facility and demand coordinates use the same abstract distance unit. The distance-decay scale is the value used in the code; no real-world geographic unit or travel-time interpretation is assumed.
- Each demand zone is evaluated independently for the service-coverage indicator. The calculation does not allocate shared facility capacity across zones, model transport flows, or enforce regulatory constraints.
- The expert feedback rules are simplified proxies. In a real decision process, experts would provide and justify their own assessments, and the final decision would remain subject to human approval.

### Partnership workflow

1. Represent facilities, demand zones, experts, and AI agents as structured data.
2. Rank facilities using the AI component's cost-benefit score and select its top four candidates.
3. Explain the recommendation and calculate service, uncertainty, and risk indicators.
4. Ask experts with different specialties to review the recommendation using explicit, illustrative rules.
5. Summarize agreement and update the simulated trust and AI-accuracy indicators.

### How to interpret the example

The generated metrics describe this programmed example only. They are not evidence of a measured improvement over human-only or AI-only decisions, and they should not be interpreted as empirical agreement, resilience, cost savings, or decision-quality claims.

## 1. Modeling the AI recommendation

The AI component assigns each facility a score that balances a simple measure of nearby demand against fixed cost. This makes the ranking easy to inspect, but it is not a proof of global optimality and does not enforce a complete network-design model.

For facility \(i\), demand zone \(j\), and selected facility set \(S\), the ranking score is

$$
R_i = \frac{b_i}{c_i}\sum_{j \in J} d_j\exp\!\left(-\frac{\ell_{ij}}{\lambda}\right),
\qquad
b_i = b(\text{importance}_i).
$$

Here, \(J\) is the set of demand zones, \(d_j\) is the base demand at zone \(j\), \(c_i\) is facility \(i\)'s fixed cost, and [0m\ell_{ij}[0m is the Euclidean distance between their coordinates. The distance-decay scale [0m\lambda[0m is 1000 in the code. The importance multiplier \(b_i\) is selected from the `strategic_bonus` mapping. The method ranks facilities by \(R_i\) and selects the four highest-scoring facilities.

The service-coverage indicator is calculated separately. For each demand zone, the best selected-facility service contribution is

$$
s_j(S)=\max_{i\in S}\left[\min(q_i,d_j)\exp\!\left(-\frac{\ell_{ij}}{\lambda}\right)\right],
\qquad
C(S)=\frac{\sum_{j\in J}s_j(S)}{\sum_{j\in J}d_j}.
$$

Here \(q_i\) is facility capacity, and \(C(S)\) is a dimensionless coverage indicator. It is not a flow-allocation solution: the same facility capacity is not jointly allocated among demand zones. Total selected fixed cost is reported alongside coverage, rather than being optimized by a formal constrained model.

In the code, `_ai_optimize_network` implements the ranking rule, while `_evaluate_solution_performance` calculates coverage and cost. Sorting \(n\) facilities takes \(O(n\log n)\) time. Generating the coverage-focused comparison enumerates all triples, taking \(O(n^3)\) selections; each selection is evaluated against the facilities and demand zones.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import Dict, List, Optional, Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## 2. Data structures

A data class keeps the fields belonging to one kind of object together. For example, a `Facility` stores its coordinates, capacity, cost, and qualitative attributes. The separate classes make the inputs to the recommendation and review steps explicit. Optional lists and dictionaries use `default_factory` so that separate instances do not accidentally share the same mutable object.

In [ ]:
@dataclass
class Facility:
    """A candidate facility and its descriptive attributes."""

    id: int
    x: float
    y: float
    capacity: int
    fixed_cost: float
    operational_status: bool = True
    strategic_importance: str = "medium"
    local_constraints: List[str] = field(default_factory=list)


@dataclass
class DemandZone:
    """A demand location with contextual attributes."""

    id: int
    x: float
    y: float
    base_demand: float
    population_served: int = 0
    vulnerability_level: str = "medium"
    access_constraints: List[str] = field(default_factory=list)


@dataclass
class HumanExpert:
    """A human reviewer with a specialty and domain-knowledge scores."""

    id: int
    name: str
    expertise_area: str
    experience_years: int
    trust_level: float = 0.5
    decision_confidence: float = 0.7
    domain_knowledge: Dict[str, float] = field(default_factory=dict)


@dataclass
class AIAgent:
    """An AI agent with a specialized capability and performance indicators."""

    id: int
    name: str
    capability: str
    processing_power: float
    explainability_score: float = 0.8
    accuracy_rate: float = 0.85
    learning_rate: float = 0.1


@dataclass
class CollaborationSession:
    """The record of one AI recommendation and human-review session."""

    session_id: int
    timestamp: datetime
    participants: List[Union[HumanExpert, AIAgent]]
    problem_context: str
    ai_recommendation: Dict
    human_feedback: Dict
    final_decision: Dict
    agreement_level: float
    trust_change: float


@dataclass
class PartnershipMetrics:
    """Summary indicators for the illustrative partnership process."""

    overall_agreement_rate: float
    decision_quality_score: float
    trust_development_score: float
    collaboration_efficiency: float
    learning_improvement_rate: float
    stakeholder_satisfaction: float

## 3. Create the example instance

The following values define a small, deterministic teaching example. Facility importance, constraints, and demand-zone context are included so that expert-review rules have relevant information to inspect. They do not automatically become hard feasibility constraints in the facility-ranking rule.

In [ ]:
def create_partnership_problem():
    """Build the facility, demand, expert, and AI-agent inputs."""

    facilities = [
        Facility(
            1, 10, 20, 1000, 200000,
            strategic_importance="high",
            local_constraints=["environmental_regulations", "union_requirements"],
        ),
        Facility(
            2, 30, 15, 1200, 250000,
            strategic_importance="critical",
            local_constraints=["transportation_limits", "zoning_restrictions"],
        ),
        Facility(
            3, 25, 35, 1100, 230000,
            strategic_importance="medium",
            local_constraints=["noise_regulations"],
        ),
        Facility(
            4, 45, 30, 1300, 280000,
            strategic_importance="medium",
            local_constraints=["access_road_limitations"],
        ),
        Facility(
            5, 15, 50, 900, 180000,
            strategic_importance="high",
            local_constraints=["flood_plain_restriction"],
        ),
        Facility(
            6, 40, 10, 1400, 300000,
            strategic_importance="critical",
            local_constraints=["high_voltage_access", "rail_proximity"],
        ),
    ]

    demand_zones = [
        DemandZone(
            1, 15, 25, 200, 50000, "high",
            access_constraints=["limited_transport_infrastructure"],
        ),
        DemandZone(
            2, 35, 20, 300, 75000, "medium",
            access_constraints=["peak_hour_congestion"],
        ),
        DemandZone(
            3, 25, 40, 250, 35000, "critical",
            access_constraints=["evacuation_route_limitation"],
        ),
        DemandZone(
            4, 40, 35, 350, 60000, "low",
            access_constraints=["seasonal_access_issues"],
        ),
    ]

    human_experts = [
        HumanExpert(
            1, "Dr. Sarah Chen", "logistics", 15, 0.5, 0.8,
            {"network_design": 0.9, "cost_optimization": 0.8, "risk_assessment": 0.7},
        ),
        HumanExpert(
            2, "Prof. Michael Rodriguez", "regulations", 20, 0.5, 0.9,
            {"environmental_law": 0.95, "zoning_regulations": 0.9, "compliance": 0.85},
        ),
        HumanExpert(
            3, "Jennifer Park", "local_knowledge", 12, 0.5, 0.7,
            {"geographic_constraints": 0.95, "community_relations": 0.9, "local_infrastructure": 0.8},
        ),
        HumanExpert(
            4, "Robert Taylor", "finance", 18, 0.5, 0.85,
            {"cost_analysis": 0.9, "investment_optimization": 0.85, "risk_management": 0.8},
        ),
        HumanExpert(
            5, "Dr. Amanda Foster", "operations", 25, 0.5, 0.9,
            {"operational_efficiency": 0.95, "process_optimization": 0.9, "resource_allocation": 0.85},
        ),
    ]

    ai_agents = [
        AIAgent(1, "OptiCore", "optimization", 0.9, 0.8, 0.88, 0.12),
        AIAgent(2, "PredictAI", "forecasting", 0.85, 0.85, 0.82, 0.15),
        AIAgent(3, "RiskSense", "risk_analysis", 0.8, 0.75, 0.90, 0.10),
    ]

    return facilities, demand_zones, human_experts, ai_agents


facilities, demand_zones, human_experts, ai_agents = create_partnership_problem()

print("Human–AI partnership example:")
print(f"Facilities: {len(facilities)}")
print(f"Demand zones: {len(demand_zones)}")
print(f"Human experts: {len(human_experts)}")
print(f"AI agents: {len(ai_agents)}")

## 4. Human–AI partnership framework

A collaboration session connects the recommendation to its review: the AI provides a facility set and explanatory indicators, each expert contributes specialty-specific feedback, and the session records a simulated agreement level. The agreement and trust calculations below are explicit proxies rather than measurements of real human behavior.

The service calculation uses Euclidean distance,

$$
\ell_{ij}=\sqrt{(x_i-x_j)^2+(y_i-y_j)^2}.
$$

The code uses this distance consistently in the recommendation, service indicator, and geographic review. Trust for each human–AI pair is updated toward the session agreement \(a\) using

$$
T_{h,a}^{\text{new}}=T_{h,a}^{\text{old}}+\eta\left(a-T_{h,a}^{\text{old}}\right),
$$

where \(T_{h,a}\) is the stored trust score and [0m\eta[0m is `trust_development_rate`. The code also updates the optimization agent's illustrative accuracy indicator toward the same session signal using the configured learning-decay factor. These updates demonstrate bookkeeping for an iterative partnership; they are not empirical learning laws.

In [ ]:
class HumanAISymbioticPartnership:
    """Coordinate explainable facility ranking and human expert review."""

    def __init__(
        self,
        facilities: List[Facility],
        demand_zones: List[DemandZone],
        human_experts: List[HumanExpert],
        ai_agents: List[AIAgent],
    ):
        self.facilities = facilities
        self.demand_zones = demand_zones
        self.human_experts = human_experts
        self.ai_agents = ai_agents
        self.collaboration_history: List[CollaborationSession] = []
        self.trust_matrix = np.full(
            (len(human_experts), len(ai_agents)), 0.5, dtype=float
        )
        self.partnership_metrics: Optional[PartnershipMetrics] = None
        self.trust_development_rate = 0.05
        self.learning_decay_rate = 0.95

    def generate_ai_recommendation(self, problem_context: str) -> Dict:
        """Rank facilities and return the recommendation with its explanations."""
        selected_facilities = self._ai_optimize_network()
        performance = self._evaluate_solution_performance(selected_facilities)
        uncertainty = self._calculate_uncertainty(selected_facilities)
        reasoning = self._generate_explainable_reasoning(
            selected_facilities, problem_context
        )

        return {
            "selected_facilities": selected_facilities,
            "reasoning": reasoning,
            "confidence": self._calculate_ai_confidence(
                performance["service_coverage"], uncertainty
            ),
            "uncertainty": uncertainty,
            "expected_performance": performance,
            "alternative_options": self._generate_alternatives(
                selected_facilities
            ),
            "risk_factors": self._identify_risk_factors(selected_facilities),
        }

    def _ai_optimize_network(self) -> List[int]:
        """Select the four facilities with the highest cost-benefit scores."""
        strategic_bonus = {
            "critical": 1.2,
            "high": 1.1,
            "medium": 1.0,
            "low": 0.9,
        }
        facility_scores = []

        for facility in self.facilities:
            if not facility.operational_status:
                continue

            service_potential = 0.0
            for demand in self.demand_zones:
                distance = self._distance(facility, demand)
                service_potential += demand.base_demand * np.exp(
                    -distance / 1000
                )

            score = (
                service_potential
                / facility.fixed_cost
                * strategic_bonus.get(facility.strategic_importance, 1.0)
            )
            facility_scores.append((score, facility.id))

        facility_scores.sort(key=lambda item: (-item[0], item[1]))
        return [facility_id for _, facility_id in facility_scores[:4]]

    @staticmethod
    def _distance(facility: Facility, demand: DemandZone) -> float:
        """Return Euclidean distance in the shared coordinate units."""
        return float(np.hypot(facility.x - demand.x, facility.y - demand.y))

    def _evaluate_solution_performance(self, selected_facilities: List[int]) -> Dict:
        """Calculate the stated coverage indicator and selected fixed cost."""
        selected = [
            facility
            for facility in self.facilities
            if facility.id in selected_facilities
        ]
        total_demand = sum(demand.base_demand for demand in self.demand_zones)
        served_demand = 0.0

        for demand in self.demand_zones:
            best_service = 0.0
            for facility in selected:
                service = min(facility.capacity, demand.base_demand) * np.exp(
                    -self._distance(facility, demand) / 1000
                )
                best_service = max(best_service, service)
            served_demand += best_service

        coverage = served_demand / total_demand if total_demand > 0 else 0.0
        return {
            "service_coverage": float(coverage),
            "total_cost": float(sum(fixed_cost.fixed_cost for fixed_cost in selected)),
            "total_capacity": int(sum(facility.capacity for facility in selected)),
            "total_demand": float(total_demand),
        }

    def _generate_explainable_reasoning(
        self, selected_facilities: List[int], context: str
    ) -> List[str]:
        """Describe the ranking basis and the selected-set indicators."""
        selected = [
            facility
            for facility in self.facilities
            if facility.id in selected_facilities
        ]
        performance = self._evaluate_solution_performance(selected_facilities)
        center_x = float(np.mean([facility.x for facility in selected]))
        center_y = float(np.mean([facility.y for facility in selected]))
        max_distance = max(
            float(np.hypot(facility.x - center_x, facility.y - center_y))
            for facility in selected
        )
        reasoning = [
            "Ranking basis: demand-weighted distance-decayed service potential, strategic-importance multiplier, and fixed cost.",
            f"Selected facilities have total fixed cost ${performance['total_cost']:,.0f}.",
            f"Selected-set service-coverage indicator: {performance['service_coverage']:.3f}.",
            f"Facility centroid: ({center_x:.1f}, {center_y:.1f}); maximum distance from centroid: {max_distance:.1f} coordinate units.",
            f"Decision context: {context}",
        ]

        critical_facilities = [
            facility.id
            for facility in selected
            if facility.strategic_importance == "critical"
        ]
        if critical_facilities:
            reasoning.append(
                f"Selected critical-importance facilities: {critical_facilities}."
            )

        constraints = [
            f"F{facility.id}: {', '.join(facility.local_constraints)}"
            for facility in selected
            if facility.local_constraints
        ]
        reasoning.append(
            "Recorded local constraints: " + "; ".join(constraints)
            if constraints
            else "Recorded local constraints: none."
        )
        return reasoning

    def _calculate_uncertainty(self, selected_facilities: List[int]) -> float:
        """Combine three normalized illustrative uncertainty components."""
        demand_variance = float(
            np.var([demand.base_demand for demand in self.demand_zones])
        )
        demand_uncertainty = min(1.0, demand_variance / 10000)

        constraint_count = sum(
            len(facility.local_constraints)
            for facility in self.facilities
            if facility.id in selected_facilities
        )
        constraint_uncertainty = min(1.0, constraint_count / 10)

        selected = [
            facility
            for facility in self.facilities
            if facility.id in selected_facilities
        ]
        pairwise_distances = [
            float(np.hypot(first.x - second.x, first.y - second.y))
            for index, first in enumerate(selected)
            for second in selected[index + 1 :]
        ]
        geographic_uncertainty = (
            min(1.0, float(np.var(pairwise_distances)) / 1000)
            if pairwise_distances
            else 0.5
        )

        return float(
            (demand_uncertainty + constraint_uncertainty + geographic_uncertainty)
            / 3
        )

    def _calculate_ai_confidence(
        self, service_coverage: float, uncertainty: float
    ) -> float:
        """Combine the optimization agent's accuracy with two indicators."""
        optimization_agent = next(
            (agent for agent in self.ai_agents if agent.capability == "optimization"),
            None,
        )
        base_confidence = (
            optimization_agent.accuracy_rate if optimization_agent else 0.8
        )
        performance_factor = min(1.0, service_coverage / 0.8)
        confidence = base_confidence * performance_factor - uncertainty * 0.2
        return float(np.clip(confidence, 0.0, 1.0))

    def _generate_alternatives(self, selected_facilities: List[int]) -> List[Dict]:
        """Report cost-focused and coverage-focused comparison selections."""
        active = [facility for facility in self.facilities if facility.operational_status]
        cost_focused = [
            facility.id
            for facility in sorted(active, key=lambda item: (item.fixed_cost, item.id))[:3]
        ]
        alternatives = []

        for name, selection in [("Cost-Focused", cost_focused)]:
            performance = self._evaluate_solution_performance(selection)
            alternatives.append(
                {
                    "name": name,
                    "facilities": selection,
                    **performance,
                }
            )

        best_selection: List[int] = []
        best_coverage = -1.0
        for first_index in range(len(active)):
            for second_index in range(first_index + 1, len(active)):
                for third_index in range(second_index + 1, len(active)):
                    selection = [
                        active[first_index].id,
                        active[second_index].id,
                        active[third_index].id,
                    ]
                    performance = self._evaluate_solution_performance(selection)
                    if performance["service_coverage"] > best_coverage:
                        best_coverage = performance["service_coverage"]
                        best_selection = selection

        performance = self._evaluate_solution_performance(best_selection)
        alternatives.append(
            {
                "name": "Coverage-Focused",
                "facilities": best_selection,
                **performance,
            }
        )
        return alternatives

    def _identify_risk_factors(self, selected_facilities: List[int]) -> List[str]:
        """List review prompts based on the selected facilities' attributes."""
        selected = [
            facility
            for facility in self.facilities
            if facility.id in selected_facilities
        ]
        risks = []

        if len(selected) >= 2:
            center_x = float(np.mean([facility.x for facility in selected]))
            center_y = float(np.mean([facility.y for facility in selected]))
            max_distance = max(
                float(np.hypot(facility.x - center_x, facility.y - center_y))
                for facility in selected
            )
            if max_distance < 30:
                risks.append(
                    "Geographic concentration merits review for regional-disruption exposure."
                )

        for facility in selected:
            if facility.local_constraints:
                risks.append(
                    f"F{facility.id} has recorded constraints: "
                    f"{', '.join(facility.local_constraints)}."
                )

        critical_count = sum(
            facility.strategic_importance == "critical" for facility in selected
        )
        if critical_count == 0:
            risks.append("No critical-importance facility is selected; review strategic coverage.")
        elif critical_count == 1:
            risks.append("One critical-importance facility is selected; review dependence on it.")

        performance = self._evaluate_solution_performance(selected_facilities)
        if performance["total_capacity"] < 0.8 * performance["total_demand"]:
            risks.append("Aggregate selected capacity is below 80% of base demand.")
        return risks

    def process_human_feedback(
        self,
        ai_recommendation: Dict,
        human_expert: HumanExpert,
        problem_context: str,
    ) -> Dict:
        """Apply the illustrative review rule for one expert specialty."""
        review_functions = {
            "regulations": self._evaluate_regulatory_compliance,
            "local_knowledge": self._evaluate_local_factors,
            "finance": self._evaluate_financial_implications,
            "operations": self._evaluate_operational_feasibility,
            "logistics": self._evaluate_logistics_optimization,
        }
        review_function = review_functions.get(
            human_expert.expertise_area, self._evaluate_logistics_optimization
        )
        feedback = review_function(ai_recommendation, human_expert)
        feedback["confidence"] = self._assess_human_confidence(
            ai_recommendation, human_expert
        )
        feedback["concerns"] = self._identify_human_concerns(
            ai_recommendation, human_expert
        )
        feedback["suggestions"] = self._generate_human_suggestions(
            ai_recommendation, human_expert, problem_context
        )
        feedback["illustrative_acceptance"] = (
            feedback["confidence"] >= 0.5 and not feedback["concerns"]
        )
        return feedback

    def _evaluate_regulatory_compliance(
        self, recommendation: Dict, expert: HumanExpert
    ) -> Dict:
        """Flag recorded constraints that lack a matching knowledge entry."""
        violations = []
        selected = set(recommendation["selected_facilities"])
        for facility in self.facilities:
            if facility.id not in selected:
                continue
            for constraint in facility.local_constraints:
                knowledge_key = constraint.replace("_", "")
                knowledge = expert.domain_knowledge.get(knowledge_key, 0.8)
                if knowledge < 0.7:
                    violations.append(f"F{facility.id}: review {constraint}.")
        return {
            "compliance_score": max(-1.0, 1.0 - 0.2 * len(violations)),
            "violations": violations,
            "recommendations": (
                ["Conduct a detailed regulatory review before implementation."]
                if violations
                else ["The illustrative rule identifies no low-knowledge constraint."]
            ),
        }

    def _evaluate_local_factors(
        self, recommendation: Dict, expert: HumanExpert
    ) -> Dict:
        """Review proximity to critical-vulnerability demand zones."""
        issues = []
        insights = []
        selected = set(recommendation["selected_facilities"])
        knowledge = expert.domain_knowledge.get("community_relations", 0.8)
        for facility in self.facilities:
            if facility.id not in selected:
                continue
            for demand in self.demand_zones:
                if (
                    self._distance(facility, demand) < 20
                    and demand.vulnerability_level == "critical"
                ):
                    if knowledge > 0.8:
                        issues.append(
                            f"F{facility.id} is close to critical-vulnerability zone D{demand.id}."
                        )
                    else:
                        insights.append(
                            f"Review community context near F{facility.id} and D{demand.id}."
                        )
        infrastructure_knowledge = expert.domain_knowledge.get(
            "local_infrastructure", 0.8
        )
        if infrastructure_knowledge > 0.7:
            for facility in self.facilities:
                if facility.id in selected and facility.x < 20 and facility.y < 30:
                    issues.append(f"F{facility.id}: review local infrastructure access.")
        return {"local_score": -0.15 * len(issues), "issues": issues, "insights": insights}

    def _evaluate_financial_implications(
        self, recommendation: Dict, expert: HumanExpert
    ) -> Dict:
        """Review fixed cost and a simple, explicitly illustrative ratio."""
        performance = recommendation["expected_performance"]
        total_cost = performance["total_cost"]
        concerns = []
        opportunities = []
        score = 0.0
        cost_knowledge = expert.domain_knowledge.get("cost_analysis", 0.8)

        if total_cost > 1000000 and cost_knowledge > 0.8:
            concerns.append(f"Review the fixed investment of ${total_cost:,.0f}.")
            score -= 0.2
        elif total_cost > 1000000:
            opportunities.append("Assess whether the service indicator supports the investment.")

        ratio = (
            performance["service_coverage"] * 0.7 / (total_cost / 1000000)
            if total_cost > 0
            else 0.0
        )
        if ratio > 0.15:
            opportunities.append(f"Illustrative coverage-to-cost ratio: {ratio:.1%}.")
            score += 0.15
        elif ratio < 0.05:
            concerns.append(f"Illustrative coverage-to-cost ratio is low: {ratio:.1%}.")
            score -= 0.15
        return {"financial_score": score, "concerns": concerns, "opportunities": opportunities}

    def _evaluate_operational_feasibility(
        self, recommendation: Dict, expert: HumanExpert
    ) -> Dict:
        """Review aggregate capacity utilization and geographic spread."""
        performance = recommendation["expected_performance"]
        selected = [
            facility
            for facility in self.facilities
            if facility.id in recommendation["selected_facilities"]
        ]
        utilization = (
            performance["total_demand"] / performance["total_capacity"]
            if performance["total_capacity"] > 0
            else float("inf")
        )
        challenges = []
        strengths = []
        score = 0.0
        if utilization > 0.9:
            challenges.append(f"Aggregate utilization indicator is {utilization:.1%}.")
            score -= 0.1
        elif utilization < 0.6:
            strengths.append(f"Aggregate utilization indicator is {utilization:.1%}.")
            score += 0.1

        if len(selected) >= 2:
            center_x = float(np.mean([facility.x for facility in selected]))
            center_y = float(np.mean([facility.y for facility in selected]))
            spread = max(
                float(np.hypot(facility.x - center_x, facility.y - center_y))
                for facility in selected
            )
            if spread > 50:
                challenges.append("Review the operational implications of wide geographic spread.")
                score -= 0.1
            elif spread < 20:
                strengths.append("The selected locations have a compact geographic spread.")
                score += 0.1
        return {
            "operational_score": score,
            "challenges": challenges,
            "strengths": strengths,
        }

    def _evaluate_logistics_optimization(
        self, recommendation: Dict, expert: HumanExpert
    ) -> Dict:
        """Measure the share of demand zones within the stated distance threshold."""
        selected = [
            facility
            for facility in self.facilities
            if facility.id in recommendation["selected_facilities"]
        ]
        if not selected or not self.demand_zones:
            connectivity = 0.0
        else:
            connected = sum(
                any(self._distance(facility, demand) < 40 for facility in selected)
                for demand in self.demand_zones
            )
            connectivity = connected / len(self.demand_zones)
        optimizations = []
        inefficiencies = []
        score = 0.0
        if connectivity > 0.8:
            optimizations.append(f"Illustrative distance connectivity: {connectivity:.1%}.")
            score += 0.2
        elif connectivity < 0.5:
            inefficiencies.append(f"Illustrative distance connectivity: {connectivity:.1%}.")
            score -= 0.2
        return {
            "logistics_score": score,
            "inefficiencies": inefficiencies,
            "optimizations": optimizations,
        }

    def _assess_human_confidence(
        self, recommendation: Dict, expert: HumanExpert
    ) -> float:
        """Calculate a bounded illustrative confidence indicator."""
        alignment = 1 - abs(
            expert.decision_confidence - recommendation["confidence"]
        )
        uncertainty_penalty = recommendation["uncertainty"] * 0.3
        explanation_bonus = min(0.2, len(recommendation["reasoning"]) / 10 * 0.2)
        alternatives_bonus = min(
            0.1, len(recommendation["alternative_options"]) / 3 * 0.1
        )
        confidence = (
            expert.decision_confidence * alignment
            - uncertainty_penalty
            + explanation_bonus
            + alternatives_bonus
        )
        return float(np.clip(confidence, 0.0, 1.0))

    def _identify_human_concerns(
        self, recommendation: Dict, expert: HumanExpert
    ) -> List[str]:
        """Generate review prompts using the illustrative thresholds."""
        concerns = []
        if recommendation["confidence"] < 0.7:
            concerns.append("Review the recommendation because its confidence indicator is below 0.7.")
        if recommendation["uncertainty"] > 0.3:
            concerns.append("Review the uncertainty indicator and its component assumptions.")
        if len(recommendation["risk_factors"]) > 2:
            concerns.append("Review the multiple risk prompts before implementation.")
        if (
            expert.expertise_area == "finance"
            and recommendation["expected_performance"]["total_cost"] > 800000
        ):
            concerns.append("Review the investment cost in greater financial detail.")
        return concerns

    def _generate_human_suggestions(
        self, recommendation: Dict, expert: HumanExpert, context: str
    ) -> List[str]:
        """Return specialty-specific prompts for further human review."""
        suggestions = {
            "regulations": "Verify applicable regulations and site-specific approvals.",
            "local_knowledge": "Check local access conditions and community context.",
            "finance": "Review fixed-cost assumptions and the coverage indicator.",
            "operations": "Validate capacity and operational assumptions with practitioners.",
            "logistics": "Validate the distance measure against actual transport conditions.",
        }
        return [
            suggestions.get(expert.expertise_area, "Review the recommendation in the problem context."),
            f"Treat the current context as a review prompt: {context}",
        ]

    def run_collaboration_session(
        self, problem_context: str, session_id: int = 1
    ) -> CollaborationSession:
        """Run one recommendation-and-review cycle and record its proxy metrics."""
        recommendation = self.generate_ai_recommendation(problem_context)
        feedback = {
            expert.name: self.process_human_feedback(
                recommendation, expert, problem_context
            )
            for expert in self.human_experts
        }
        acceptance_values = [
            item["illustrative_acceptance"] for item in feedback.values()
        ]
        agreement = (
            float(np.mean(acceptance_values)) if acceptance_values else 0.0
        )
        old_trust = float(np.mean(self.trust_matrix)) if self.trust_matrix.size else 0.0
        self.trust_matrix += self.trust_development_rate * (
            agreement - self.trust_matrix
        )
        new_trust = float(np.mean(self.trust_matrix)) if self.trust_matrix.size else 0.0
        trust_change = new_trust - old_trust

        optimization_agent = next(
            (agent for agent in self.ai_agents if agent.capability == "optimization"),
            None,
        )
        if optimization_agent is not None:
            optimization_agent.accuracy_rate = float(
                np.clip(
                    self.learning_decay_rate * optimization_agent.accuracy_rate
                    + (1 - self.learning_decay_rate) * agreement,
                    0.0,
                    1.0,
                )
            )

        session = CollaborationSession(
            session_id=session_id,
            timestamp=datetime.now(),
            participants=self.human_experts + self.ai_agents,
            problem_context=problem_context,
            ai_recommendation=recommendation,
            human_feedback=feedback,
            final_decision={
                "status": "pending_human_approval",
                "candidate_facilities": recommendation["selected_facilities"],
                "note": "The simulation does not make the final implementation decision.",
            },
            agreement_level=agreement,
            trust_change=trust_change,
        )
        self.collaboration_history.append(session)
        self.partnership_metrics = self.calculate_partnership_metrics()
        return session

    def calculate_partnership_metrics(self) -> PartnershipMetrics:
        """Summarize recorded sessions without treating proxies as empirical data."""
        if not self.collaboration_history:
            return PartnershipMetrics(0.0, 0.0, 0.0, 0.0, 0.0, 0.0)

        sessions = self.collaboration_history
        agreement = float(np.mean([item.agreement_level for item in sessions]))
        quality = float(
            np.mean(
                [
                    item.ai_recommendation["expected_performance"]["service_coverage"]
                    for item in sessions
                ]
            )
        )
        trust = float(np.mean(self.trust_matrix)) if self.trust_matrix.size else 0.0
        efficiency = 1.0 / (1.0 + len(self.human_experts))
        learning = float(
            np.mean([agent.accuracy_rate for agent in self.ai_agents])
        )
        satisfaction = agreement
        return PartnershipMetrics(
            overall_agreement_rate=agreement,
            decision_quality_score=quality,
            trust_development_score=trust,
            collaboration_efficiency=efficiency,
            learning_improvement_rate=learning,
            stakeholder_satisfaction=satisfaction,
        )

## 5. Run and inspect a collaboration session

The session returns an AI recommendation and expert feedback, but leaves the implementation decision pending human approval. This separation matters: the code can summarize a decision process, but its programmed acceptance rule is not a substitute for informed human judgment. The assertions check internal consistency of the recommendation and its reported cost.

In [ ]:
partnership = HumanAISymbioticPartnership(
    facilities=facilities,
    demand_zones=demand_zones,
    human_experts=human_experts,
    ai_agents=ai_agents,
)

session = partnership.run_collaboration_session(
    problem_context="Illustrative southeastern US network redesign",
    session_id=1,
)

recommendation = session.ai_recommendation
selected_ids = recommendation["selected_facilities"]
selected_cost = sum(
facility.fixed_cost
for facility in facilities
if facility.id in selected_ids
)

assert len(selected_ids) == 4
assert len(set(selected_ids)) == len(selected_ids)
assert set(selected_ids).issubset({facility.id for facility in facilities})
assert np.isclose(
    selected_cost,
    recommendation["expected_performance"]["total_cost"],
)
assert 0.0 <= session.agreement_level <= 1.0
assert session.final_decision["status"] == "pending_human_approval"

print("AI-selected facility IDs:", selected_ids)
print(
    "Service-coverage indicator:",
    f"{recommendation['expected_performance']['service_coverage']:.3f}",
)
print(
    "Selected fixed cost:",
    f"${recommendation['expected_performance']['total_cost']:,.0f}",
)
print("Illustrative uncertainty:", f"{recommendation['uncertainty']:.3f}")
print("Illustrative session agreement:", f"{session.agreement_level:.1%}")
print("Final-decision status:", session.final_decision["status"])
print("\nAI explanation:")
for statement in recommendation["reasoning"]:
    print("-", statement)

print("\nExpert review summary:")
for expert_name, feedback in session.human_feedback.items():
    print(
        f"- {expert_name} ({feedback['confidence']:.2f} confidence proxy): "
        f"{len(feedback['concerns'])} concern(s), "
        f"{len(feedback['suggestions'])} review prompt(s)"
    )

print("\nPartnership metrics:")
for metric_name, metric_value in vars(partnership.partnership_metrics).items():
    print(f"- {metric_name}: {metric_value:.3f}")

## 6. Interpreting the results

- **Selected facility IDs** are the four highest-ranked active facilities under the cost-benefit heuristic. They are not a proof of an optimal or robust network design.
- **Service coverage** is the distance-decayed indicator defined above. It does not represent allocated flows, unmet demand under shared capacity, or a guarantee of service.
- **Uncertainty and confidence** are constructed indicators based on the example's scaling rules and the optimization agent's configured accuracy. They are not calibrated probabilities.
- **Expert feedback** is generated by deterministic specialty-specific rules. In particular, the session's agreement rate is the share of simulated reviewers whose confidence proxy reaches the illustrative threshold without rule-generated concerns; it is not a survey or observed human–AI agreement rate.
- **Trust and learning metrics** summarize the state updates coded in the session. They should be interpreted as properties of this simulation, not evidence that a real partnership will improve over time.
- **Final-decision status** remains pending human approval. The model surfaces information for review; it does not authorize facility selection or implementation.

### Partnership considerations

Human–AI collaboration can be useful when decisions require contextual expertise, explanation, and accountable review. It also introduces coordination costs and risks, including human bias, automation bias, uneven expertise, and misunderstandings about what a model's indicators mean. A practical deployment would require validated data, explicit decision authority, documented expert input, and evaluation against an appropriate real-world objective.

This example deliberately omits real travel networks, demand allocation, facility-opening decisions under formal constraints, uncertain demand scenarios, and empirical measures of human performance. Those features would require additional problem data and explicit modeling choices; the indicators in this notebook do not stand in for them.